# Qwen3-VL 8B: does text steering transfer to captions in the image?

**Start:** Select Kernel > Colab > New Colab Server > GPU > **A100**, or **L4** if no A100 (24 GB fits; slower, cheaper per hour). Right-click `qwen_transfer_bundle.tar.gz` in the explorer > **Upload to Colab**. Command Palette > **Colab: Mount Google Drive to Server**. Then run the cells top to bottom. File > Auto Save on, so Claude can read the output.

**Cost:** about 5.5 to 7 units per hour. Estimated 2 to 4 hours in total. If Colab disconnects, run the cells again; it continues. **When done: Colab: Remove Server.**

**On the Colab website instead:** File > Upload notebook (this file). Runtime > Change runtime type > A100 (or L4). Put `qwen_transfer_bundle.tar.gz` in Google Drive at `My Drive/idea3_qwen_transfer/`. The first cell asks to connect Drive. At the end, the zip cell makes one file to send back.

Nothing here can run the test scenes.

In [ ]:
import os, subprocess
print(subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; df -h /content | tail -1; python3 --version; ls /content', shell=True, capture_output=True, text=True).stdout)
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive; drive.mount('/content/drive')   # Colab website; in VS Code use the Command Palette instead
    except Exception as ex:
        print('Drive not mounted here:', ex)
print('Drive mounted:', os.path.exists('/content/drive/MyDrive'))

In [ ]:
%%writefile /content/qt_colab.py
"""Colab helper for the Qwen3-VL transfer test. Plain Python. The experiment runs in its own uv environment (torch 2.9.1 + transformers
4.57.6 + pillow 12.3.0). Resumable: results are copied to Drive every minute and restored on start. Test scenes stay locked."""
import json, os, re, shutil, subprocess, threading, time
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/idea3_qwen_transfer') if Path('/content/drive/MyDrive').exists() else Path('/content/qt_state')
STATE, ROOT, CODE, VENV = DRIVE / 'results', Path('/content/qt'), Path('/content/qt/qwen_transfer'), Path('/content/qtenv')
PY, HF, P777 = VENV / 'bin/python', Path('/content/hf'), Path('/content/qt/p777')
BUNDLE = 'qwen_transfer_bundle.tar.gz'
REV = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'


def sh(cmd, **k):
    print('$', cmd if isinstance(cmd, str) else ' '.join(map(str, cmd)), flush=True); subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **k)


def env(stop_at=None):
    e = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME', 'MPLBACKEND', 'ALLOW_TEST')}
    e.update(P777=str(P777), DATA_DIR=str(P777 / 'data/egonormia'), REF_DIR=str(P777 / 'data/ref'), HF_HOME=str(HF), HF_HUB_OFFLINE='1',
             TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1', MPLBACKEND='Agg', PATH=f'{VENV}/bin:' + os.environ['PATH'])
    if stop_at:
        e['STOP_AT'] = str(stop_at)
    return e


def gpu_check(min_gb=22):
    out = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits', shell=True, capture_output=True, text=True).stdout.strip()
    print('GPU:', out or 'NONE'); assert out, 'No GPU. Choose L4 or A100 and connect again.'
    assert int(out.splitlines()[0].split(',')[1]) > min_gb * 1000, 'GPU is smaller than expected (need at least 24 GB: L4 or A100); disconnect so it does not use units.'


def copy_new(src, dst):
    n = 0
    for p in src.rglob('*') if src.exists() else []:
        if p.is_dir() or p.suffix == '.tmp':
            continue
        q = dst / p.relative_to(src); s = p.stat()
        if q.exists() and q.stat().st_size == s.st_size and abs(q.stat().st_mtime - s.st_mtime) < 2:
            continue
        q.parent.mkdir(parents=True, exist_ok=True); t = q.with_name(q.name + '.part'); shutil.copy2(p, t); os.replace(t, q); os.utime(q, (s.st_mtime, s.st_mtime)); n += 1
    return n


def sync():
    return copy_new(CODE / 'results', STATE)


def setup():
    if not str(DRIVE).startswith('/content/drive'):
        print('WARNING: Drive is not mounted: progress is LOST on disconnect. Command Palette > "Colab: Mount Google Drive to Server", then run from the top.')
    DRIVE.mkdir(parents=True, exist_ok=True)
    if not (ROOT / '.unpacked').exists():
        found = [c for c in (Path('/content') / BUNDLE, DRIVE / BUNDLE) if c.exists()]
        assert found, f'Upload {BUNDLE} first (right-click it in VS Code > "Upload to Colab").'
        ROOT.mkdir(parents=True, exist_ok=True); sh(['tar', 'xzf', str(found[0]), '-C', str(ROOT)])
        if not (DRIVE / BUNDLE).exists():
            shutil.copy(found[0], DRIVE / BUNDLE)
        if STATE.exists():
            print('restored from Drive:', copy_new(STATE, CODE / 'results'), 'files')
        (ROOT / '.unpacked').touch()
    if not (VENV / '.ok').exists():
        sh('pip -q install uv'); sh(f'uv venv -q --clear --python 3.12 {VENV}')
        sh(f'uv pip install -q --python {PY} --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu128 torch==2.9.1+cu128 torchvision==0.24.1+cu128 transformers==4.57.6 "accelerate>=1.0" pillow==12.3.0 pyyaml matplotlib numpy requests huggingface_hub hf_xet')
        (VENV / '.ok').touch()
    e = env()
    if not (P777 / 'data/.images_ok').exists():
        sh([str(PY), '-I', 'src/download_data.py', '--workers', '32'], cwd=CODE, env=e)
        rep = json.loads((P777 / 'data/egonormia/verify_report.json').read_text())
        assert rep['raw_sha_ok'] == 1853 and rep['preview_sha_ok'] == 1853, rep
        (P777 / 'data/.images_ok').touch()
    if not (HF / '.weights_ok').exists():
        e2 = dict(env(), HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0')
        print('Downloading the model (about 17.5 GB)', flush=True)
        sh([str(PY), '-c', f"from huggingface_hub import snapshot_download as s; print(s('Qwen/Qwen3-VL-8B-Instruct', revision='{REV}', max_workers=16))"], env=e2); (HF / '.weights_ok').touch()
    out = subprocess.run([str(PY), '-c', 'import torch, transformers, PIL; print(torch.__version__, transformers.__version__, PIL.__version__, torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())'], env=env(), capture_output=True, text=True)
    print('environment:', out.stdout.strip() or out.stderr[-600:]); assert out.returncode == 0; print('SETUP DONE')


def stage(name, end):
    log = CODE / 'results/colab_log.txt'; print(f'=== {name}', time.strftime('%H:%M'), flush=True)
    p = subprocess.Popen([str(PY), 'src/transfer.py', *name.split()], cwd=CODE, env=env(end), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    with open(log, 'a') as fh:
        for line in p.stdout:
            fh.write(line); fh.flush()
            if not re.search(r'it/s|Loading checkpoint|FutureWarning|UserWarning|^\s*$', line):
                print(line.rstrip()[:420], flush=True)
    p.wait(); print(f'=== {name} exit {p.returncode}', time.strftime('%H:%M'), flush=True); return p.returncode


def run(stages, hours=8.0):
    """Runs the stages in order, resumably, until done or the time limit. Stops at the first stage that fails or stops itself."""
    end = time.time() + hours * 3600; stop_sync = threading.Event()
    def loop():
        while not stop_sync.wait(60):
            try: sync()
            except Exception as ex: print('sync problem (will retry):', ex, flush=True)
    threading.Thread(target=loop, daemon=True).start()
    try:
        for s in stages:
            code = stage(s, end)
            if (CODE / 'results/STOP.md').exists():
                print('STOPPED BY A GATE:', (CODE / 'results/STOP.md').read_text()); break
            if code == 75 or time.time() > end - 120:
                print('time limit reached; run this cell again to continue'); break
            if code:
                print('stage failed; send this output to Claude'); break
        else:
            print('ALL STAGES IN THIS CELL DONE')
    finally:
        stop_sync.set(); time.sleep(1)
        for _ in range(3):
            if sync() == 0: break
        print('copied to Drive:', STATE)


def release():
    print('Releasing the GPU'); time.sleep(30)
    try:
        from google.colab import runtime; runtime.unassign()
    except Exception as ex:
        print('Could not release automatically: Command Palette > "Colab: Remove Server" NOW.', ex)


In [ ]:
import sys; sys.path.insert(0, '/content')
import qt_colab as q
q.gpu_check()   # stops here if the GPU is wrong
q.setup()       # about 10 to 20 minutes the first time

## Part 1: checks and base bias (no steering). About 1 to 1.5 hours
1. reproduce midterm text answers (must match 98%)  2. can it read the caption (90%)  3. base answers on image captions  4. bias gate

In [ ]:
q.run(['repro', 'perc', 'base', 'gate'], hours=4)

## Part 2: the steered runs
Read the success rule in `QWEN_TRANSFER_PLAN.md`. **Only if you agree, set the flag below to True.**

In [ ]:
RULE_APPROVED = False      # set to True only after Yash has read and approved the success rule
if RULE_APPROVED:
    from pathlib import Path
    Path('/content/qt/qwen_transfer/results/PREREG_OK').write_text('approved by Yash\n')
    q.run(['exp', 'report val:tr'], hours=6)
else:
    print('Not started: set RULE_APPROVED = True after reading the rule.')

In [ ]:
# optional: see the tables any time
q.stage('gate', None); q.stage('report val:tr', None)

In [ ]:
# when finished: release the GPU so it stops using units
q.sync(); q.release()

In [ ]:
# send-back file: zips all results (no images, no model). Saved to Drive (My Drive/idea3_qwen_transfer/) and /content
import shutil, time
z = shutil.make_archive(f'/content/qt_results_{time.strftime("%m%d_%H%M")}', 'zip', '/content/qt/qwen_transfer/results')
if os.path.exists('/content/drive/MyDrive'): shutil.copy(z, '/content/drive/MyDrive/idea3_qwen_transfer/')
print('send this file:', z, round(os.path.getsize(z) / 1e6, 1), 'MB')